[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/08_Intro_redes_neuronales.ipynb)


# Redes neuronales — empezando por algo muy chico

En muchos cursos aparece GPT o el diagnóstico médico… aquí resolvemos algo mínimo: **predecir estatura desde el tamaño de la mano**.

Primero hacemos una **recta** (regresión lineal). Es rápido y claro.

Después mostramos que **podemos describir ese mismo trabajo como una red pequeña** y entrenarla con **TensorFlow**. Así se ve el puente antes de redes más locas donde ya no obligamos una relación lineal.

Más adelante: **clasificación** con **Rain in Australia** (`RainTomorrow`, curvas de pérdida y accuracy), el ejemplo de **puntos rojos y azules** (`make_moons`), **tres clases con softmax**, una tabla de **hiperparámetros** y un **ejercicio** para experimentar.


In [ ]:
# Instala todo lo que se usa a lo largo del notebook (descomenta la línea si te falta algo).
# %pip install -q numpy matplotlib pandas scikit-learn tensorflow gdown ipython

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})


## 1 — Regresión lineal (una recta)

La idea: $\hat{y} = \beta_1 + \beta_2\,x$ con $x$ = tamaño de la mano y $\hat{y}$ = estatura predicha. **$\beta_1$** es el intercepto y **$\beta_2$** la pendiente. OLS busca la recta que mejor cuadra con los puntos (en el sentido de minimizar errores al cuadrado).

Siguiente celda: ajustar con **scikit-learn** sobre los **siete puntos**; después predices un valor nuevo y ves el gráfico.


In [ ]:
from sklearn.linear_model import LinearRegression

# --- Datos ejemplo (diagrama típico mano ↔ estatura) ---
x_mano = np.array([13.0, 16.0, 24.0, 43.0, 51.0, 84.0, 90.0])
y_estatura = np.array([23.0, 5.0, 33.0, 32.0, 53.0, 65.0, 85.0])

# sklearn quiere cada fila con sus propias características → matriz `(n_filas, 1)`.
ols_1var = LinearRegression().fit(x_mano.reshape(-1, 1), y_estatura)

# Coeficientes leíbles para discutir la forma β₁ + β₂·mano.
beta1 = float(ols_1var.intercept_)
beta2 = float(ols_1var.coef_[0])

print('OLS con siete puntos — coeficientes estimados:')
print(f'  β₁ (intercepto):  {beta1:.4f}')
print(f'  β₂ (pendiente):   {beta2:.4f}')


In [ ]:
# Puedes mover `mano_nueva` cualquier tamaño físico nuevo que quieran simular tras entrenamiento.
mano_nueva = 55.0

row = np.array([[mano_nueva]])   # igual que arriba, la API exige formato fila‑columnas
estatura_pred = float(ols_1var.predict(row)[0])

print(f'Si el tamaño de la mano es {mano_nueva:g}:')
print(f'  → estatura predicha ≈ {estatura_pred:.2f}')
print(
    '(comprobación manual: β₁ + β₂×mano = '
    f'{beta1:.4f} + {beta2:.4f} × {mano_nueva:g} ≈ '
    f'{beta1 + beta2 * mano_nueva:.2f})'
)

In [ ]:
# Curva interpolada muy suave sólo muestra mejor la línea encontrada contra los dispersos violetas.
fine = np.linspace(float(x_mano.min()), float(x_mano.max()), 200)
recta = beta1 + beta2 * fine

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_mano, y_estatura, alpha=0.88, color='#7C3AED', edgecolors='white',
           linewidths=0.5, label='Las 7 observaciones')
ax.plot(fine, recta, color='#DC2626', linewidth=2.5,
        label='Recta ajustada: β₁ + β₂ · tam_mano')

ax.set_xlabel('Tamaño de la mano')
ax.set_ylabel('Estatura')
ax.set_title('Mínimos cuadrados sobre siete puntos')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## 2 — Misma historia, vista como red neuronal

En libros aparece que muchos procedimientos se pueden dibujar como **red**. La idea práctica aquí es: **mis datos son los mismos** (filas de mano ↔ estatura), pero en vez de la fórmula cerrada del OLS vas a **entrenar**: el programa ajusta **pesos** repitiendo.

Cuando ya entrenaste, llega una persona nueva con su medida de mano y preguntas: **¿qué estatura predigo?** — igual que con la recta, pero después de esta rutina llamada gradiente/épocas que verás correr abajo.

Más adelante veremos redes donde **ya no obligamos una recta**. Hoy todas las activaciones son **lineales** a propósito: la familia del modelo sigue siendo cercana al caso lineal para que pegue con el paso 1. **Entrada**: un número (tamaño de mano).

En código: **entrada** = un número por fila (la mano) → **capa oculta** `Dense` con activación **lineal** → **salida** `Dense(1)` también **lineal**. Así ves la “cascarita” entrada–medio–salida como en los diagramas generales del curso.

In [ ]:
# TensorFlow y Keras: aquí sólo cargamos el ecosistema. Las dos celdas siguientes
# prepararán datos y crearán capas del modelo paso por paso.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# Cada FILA es un ejemplo: una columna = la mano (`float32` es lo que suele esperar la red).
# `y_nn` lleva las estaturas observadas pareadas por índice con `x_mano`.
X_nn = np.asarray(x_mano, dtype=np.float32).reshape(-1, 1)
y_nn = np.asarray(y_estatura, dtype=np.float32)

tf.keras.utils.set_random_seed(42)

In [ ]:
# `Sequential`: se leen capas como un tubo izquierda → derecha.
# Una entrada escalar amplificada primero a 8 salidas LINEALES,
# después se combina todo en una sola predicción (también lineal): sigue pareciendo a una recta.
modelo_mano = keras.Sequential(
    [
        layers.Input(shape=(1,), name='entrada_mano'), # el shape es el numero de caracteristicas, y como sólo tenemos el tamaño de la mano, es 1
        layers.Dense(8, activation='linear', name='capa_oculta'),
        layers.Dense(1, activation='linear', name='salida_estatura'),
    ],
    name='ejemplo_recta_lineal',
)

In [ ]:
# `compile`: eliges cómo mejorar pesos (`optimizer`) y qué pérdida bajar durante el entrenamiento (`mse`).
modelo_mano.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    loss='mse',
)

### Aparte: ¿qué hace `Adam` y en qué se diferencia de otros optimizadores?

Cada vez que pones `optimizer=...` en `compile`, estás eligiendo **la receta** con la que la red va a usar los gradientes para mover los pesos. Todos los optimizadores hacen lo mismo conceptualmente — `peso_nuevo = peso_viejo − algo × gradiente` — pero **ese "algo"** se calcula distinto en cada uno, y eso cambia drásticamente la velocidad y estabilidad del entrenamiento.

**Idea central de Adam:** combina dos trucos famosos en un solo optimizador.

1. **Momento (memoria de la dirección)**: en vez de moverse exactamente en la dirección del gradiente actual, Adam **promedia** los gradientes recientes — como si la pelotita tuviera **inercia**. Si va bajando hacia el valle y aparece un gradiente raro de un lote ruidoso, no le hace tanto caso porque el promedio sigue apuntando al valle.

2. **Learning rate adaptativo por parámetro**: Adam lleva un **historial de cuánto ha vibrado el gradiente de cada peso individualmente**. A los pesos cuyos gradientes son ruidosos o grandes les da pasos chicos; a los pesos con gradientes consistentemente pequeños les da pasos más grandes. Es como tener **un learning rate personalizado para cada uno de los parámetros**, calculado automáticamente.

Ese par de trucos lo hace muy robusto: con configuración default funciona en casi cualquier problema. Por eso es **el "default razonable"** de la industria.

| Optimizador | ¿Qué hace? | Pros | Contras | Cuándo usarlo |
|---|---|---|---|---|
| **SGD** (vanilla) | El más básico: `peso − lr × gradiente`, punto | Simple, predecible, generaliza bien al final | Lento, ruidoso, sensible al `learning_rate`, se atora en valles planos | Cuando ya tienes hiperparámetros bien tuneados (ej. ImageNet con scheduling fino) |
| **SGD + Momentum** | SGD con "inercia": acumula velocidad en la dirección que viene siguiendo | Suaviza el ruido, atraviesa valles planos | Sigue requiriendo tunear lr y el momento | Visión por computadora clásica (ResNet, etc.) |
| **RMSprop** | Adapta el lr por parámetro según el historial de gradientes al cuadrado | Bueno para redes recurrentes, robusto al lr | No tiene momento "real" | RNNs, problemas con gradientes muy variables |
| **Adam** | Momentum + RMSprop combinados | Robusto out-of-the-box, converge rápido, casi siempre funciona razonable | A veces generaliza un poco peor que SGD bien tuneado | **Default seguro** para casi todo |
| **AdamW** | Adam con regularización de pesos (`weight decay`) corregida | Mejor generalización que Adam vanilla en redes grandes | Casi igual de complejo que Adam | Transformers, modelos grandes (BERT, GPT, ViT) |
| **Adagrad** | lr por parámetro que **decrece** con el tiempo | Bueno para datos sparse (NLP clásico) | El lr llega a casi 0 — el modelo deja de aprender | Word embeddings, datos sparse |
| **Nadam** | Adam + Nesterov momentum (un momentum más "anticipativo") | A veces converge un pelín más rápido que Adam | Diferencia chica en la mayoría de casos | Casi intercambiable con Adam |

**Reglas prácticas:**

- Si no sabes qué usar, **empieza con Adam** y `learning_rate=0.001` (el default). Es honesto: es lo que hace casi toda la industria al prototipar.
- Si vas a entrenar un **transformer** o un modelo grande, usa **AdamW**.
- Si quieres **squeeze de generalización en visión**, prueba **SGD + Momentum** con un learning rate scheduler (cosine decay, warm-up) — históricamente le ha ganado a Adam en benchmarks de imagen.
- **No cambies el optimizador** como primera medida cuando algo no funciona. Antes revisa: tasa de aprendizaje, escalado de datos, arquitectura, cantidad de datos. El optimizador rara vez es el cuello de botella.

> En este cuaderno usamos `Adam` para todos los modelos justo por eso: con un `learning_rate` razonable funciona sin necesidad de afinar nada más. Si más adelante quieres experimentar, cambia a `keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)` y compara las curvas de pérdida — vas a notar diferencias sobre todo en velocidad de convergencia.


`fit` = enseñar el modelo repetidas veces con los mismos pares *(mano, estatura)* y bajar la pérdida `mse` (error cuadrático medio). Ahí aparece la idea moderna del entrenamiento: medir mal desempeño, calcular gradientes, actualizar.

In [ ]:
# `epochs`: cuántas veces recorrerá el conjunto completo antes de frenar esta corrida rápida.
# `verbose=0` sólo oculta la barrita ruidosa; la historia queda igual en la variable Python.
historia = modelo_mano.fit(X_nn, y_nn, epochs=800, verbose=0)

print('Entrenamiento listo.')
print(f"Última pérdida (mse): {float(historia.history['loss'][-1]):.6f}")

### ¿Qué es ese número de `mse`?

En entrenamiento, TensorFlow está minimizando el **error cuadrático medio** sobre las **filas que le pasaste** (aquí, 7 puntos). Para cada fila $i$:

$$
e_i^2 = (y_i - \hat{y}_i)^2
$$

donde $e_i$ es el residual (error de predicción) de esa fila.

La pérdida que reporta `mse` es la media:

$$
\mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

Aquí $n=7$.

En forma compacta:

$$
h = xW^{(1)} + b^{(1)}, \qquad \hat{y} = hW^{(2)} + b^{(2)}.
$$

Como todas las capas aquí son lineales y $x$ es escalar, toda la red se reduce a una recta efectiva:

$$
\hat{y} = c_0 + c_1 x.
$$

Por eso, aunque internamente haya muchos parámetros, el comportamiento final frente a $x$ puede compararse con la pendiente de OLS.

In [ ]:
# Extraemos pesos internos de las capas Dense para poder colapsar la red lineal a una recta.
densas_tf = [capa for capa in modelo_mano.layers if isinstance(capa, keras.layers.Dense)]
capa_oculta, capa_salida = densas_tf[0], densas_tf[1]
W1, b1 = capa_oculta.get_weights()
W2, b2 = capa_salida.get_weights()

In [ ]:
# Composición lineal de la red -> pendiente efectiva única.
pendiente_ef = float(np.dot(W1.ravel(), W2.ravel()))

print('Recta efectiva de la red lineal (resumen):')
print(f'  c1 (pendiente efectiva de la red): {pendiente_ef:.6f}')

print('\nComparación con regresión lineal (OLS):')
print(f'  β₂ OLS (pendiente):                {beta2:.6f}')

Misma persona nueva (`mano_nueva`): comparamos predicción por **recta (OLS)** y por **red**. Con capas lineales, ambas deberían dar casi el mismo valor para el mismo dato.

In [ ]:
# Inferencia: mismo formato que en sklearn — un escalar se empaqueta en matriz (1, 1) tipo float32.
mano_tf = np.array([[mano_nueva]], dtype=np.float32)
estatura_tf = float(modelo_mano.predict(mano_tf, verbose=0)[0, 0])

print(f'Mano nueva = {mano_nueva:g}')
print(f'  Estatura (OLS / recta):      {estatura_pred:.4f}')
print(f'  Estatura (red TF lineal):    {estatura_tf:.4f}')

La regresión lineal fue "en un paso"; la red lineal de TensorFlow repitió el mismo tipo de respuesta.

Ahora pasamos a **clasificación**: ahí tiene sentido mirar **pérdida** y **precisión** (*accuracy*) a lo largo de las **épocas**. Primero el dataset **Rain in Australia** (muchas columnas numéricas → ¿lloverá mañana?); después el clásico de **puntos rojos y azules** en el plano.

## 3 — Lluvia en Australia (`RainTomorrow`)

Usamos el dataset público **Rain in Australia** (Kaggle: *Weather dataset / Rattle package*). Objetivo: **`RainTomorrow`** (Yes/No → lluvia al día siguiente según la definición del dataset).

En este cuaderno el CSV se descarga desde [Google Drive — `weatherAUS.csv`](https://drive.google.com/file/d/1QwlzYISF4V8vDYSINoCu4X72A_bBJrgX/view?usp=sharing) usando el `DRIVE_FILE_ID` de la celda **3.1**.

Columnas típicas del CSV: **Date**, **Location**, temperaturas (**MinTemp**, **MaxTemp**, **Temp9am**, **Temp3pm**), **Rainfall**, **Evaporation**, **Sunshine**, viento (**WindGustDir**, **WindGustSpeed** u homónimos, **WindDir9am**, **WindDir3pm**, **WindSpeed9am**, **WindSpeed3pm**), **Humidity9am**, **Humidity3pm**, **Pressure9am**, **Pressure3pm**, **Cloud9am**, **Cloud3pm**, **RainToday**, **RainTomorrow**.

En las siguientes celdas **separamos** configuración, lectura, limpieza, partición, escalado y gráficos para que no quede todo amontonado.

### 3.1 — ID del archivo en Google Drive

Aquí va el **ID** del `weatherAUS.csv` compartido (la celda **3.2** lo descarga con `gdown`).


In [ ]:
# ID del archivo .csv de drive
DRIVE_FILE_ID = "1QwlzYISF4V8vDYSINoCu4X72A_bBJrgX"  # weatherAUS.csv (Drive compartido)

# Nombre temporal si descargas con gdown
NOMBRE_DESCARGA = "weatherAUS.csv"


### 3.2 — Descarga y lectura del CSV

Instala **`gdown`** una vez si hace falta: `%pip install gdown -q`.


In [ ]:
import pandas as pd
import gdown

url = f"https://drive.google.com/uc?id={DRIVE_FILE_ID}"
gdown.download(url, NOMBRE_DESCARGA, quiet=False)

df = pd.read_csv(NOMBRE_DESCARGA)
print("Filas, columnas:", df.shape)
df.head(3)


### 3.3 — Objetivo: `RainTomorrow`

Comprobamos que exista la columna y la pasamos a **0 = No**, **1 = Yes**.


In [ ]:
COL_OBJETIVO = "RainTomorrow"

if COL_OBJETIVO not in df.columns:
    raise ValueError(f"No está la columna {COL_OBJETIVO}. Revisa el CSV.")

y_raw = df[COL_OBJETIVO].map({"Yes": 1.0, "No": 0.0})
print("Valores nulos en y:", int(y_raw.isna().sum()))


### 3.4 — Columnas numéricas que usaremos como X

Dejamos fuera el viento en **texto** (direcciones). El CSV de Kaggle suele usar **`WindGustSpeed`** para la ráfaga; si viene otro nombre, lo detectamos.


In [ ]:
gust_col = (
    "WindGustSpeed"
    if "WindGustSpeed" in df.columns
    else ("WindGustSpd" if "WindGustSpd" in df.columns else None)
)
cols_num = [
    "MinTemp", "MaxTemp", "Rainfall", "Evaporation", "Sunshine",
    "WindSpeed9am", "WindSpeed3pm",
    "Humidity9am", "Humidity3pm",
    "Pressure9am", "Pressure3pm",
    "Cloud9am", "Cloud3pm",
    "Temp9am", "Temp3pm",
]
if gust_col:
    cols_num = [gust_col] + cols_num

cols_presentes = [c for c in cols_num if c in df.columns]
faltan = sorted(set(cols_num) - set(cols_presentes))
if faltan:
    print("Aviso: no aparecen en el CSV (se omiten):", faltan)
print("Columnas que entrarán a X:", cols_presentes)


### 3.5 — Quitar filas con NaN en X o en y

Coercionamos a número y nos quedamos sólo con filas **completas** en todas las features y con etiqueta válida.


In [ ]:
X_raw = df[cols_presentes].apply(pd.to_numeric, errors="coerce") # Coerce remplaza por NaNs
mask = y_raw.notna() & X_raw.notna().all(axis=1)
X_df = X_raw.loc[mask].reset_index(drop=True)
y_series = y_raw.loc[mask].reset_index(drop=True)

print("Filas tras limpiar NaN:", len(X_df))
print("Proporción RainTomorrow=1:", float(y_series.mean()))


### 3.6 — Partición train / validación

Separamos **antes** de escalar o de entrenar. Usamos **estratificación** en `y` para que train y val tengan proporción parecida de lluvia.


In [ ]:
from sklearn.model_selection import train_test_split

X_train_df, X_val_df, y_train_s, y_val_s = train_test_split(
    X_df,
    y_series,
    test_size=0.2,        # 80% para entrenar y 20% para validar
    random_state=42,      # semilla: misma partición cada vez que corras
    stratify=y_series,    # estratificar = mantener la misma proporción de "llueve / no llueve" en train y en val
)

print("Train:", X_train_df.shape, "| Val:", X_val_df.shape)


### 3.7 — Escalado (`StandardScaler`)

**Escalar vs. normalizar.** En la práctica los nombres se confunden, pero el uso común es:

- **Escalar / estandarizar** (`StandardScaler`): a cada columna le restas su **media** y divides entre su **desviación estándar**. Queda con media ≈ 0 y desv. ≈ 1 (es el *z‑score*). No fuerza un rango fijo.
- **Normalizar min‑max** (`MinMaxScaler`): llevas cada columna al rango **[0, 1]** usando su mínimo y máximo.

> Ojo: `Normalizer` de sklearn es otra cosa — normaliza **filas** a norma 1, útil para texto/coseno, no para tabular.

**¿Por qué escalamos aquí?** Las columnas del CSV viven en **escalas muy distintas**: `Pressure9am` ronda 1000, `Humidity3pm` va de 0 a 100, `Rainfall` puede llegar a 300, `Cloud3pm` va de 0 a 8. Si las metes crudas a la red:

- El optimizador (gradient descent) avanza muy desbalanceado: los pesos asociados a presión deben quedar **diminutos** y los de nubes **enormes** para compensar las escalas. Eso hace que la pérdida baje despacio o se atore.
- Con `StandardScaler` todas las features quedan en un rango parecido (~ ‑3 a +3), y los pesos parten más simétricos. El entrenamiento converge **más rápido y estable**.
- Preferimos estandarizar sobre min‑max porque es **más robusto a outliers** (un valor extremo no aplasta el resto contra 0).

**¿Por qué `fit` solo en train?** La media y la desviación se calculan **solo con el train**; al `X_val` se le aplica el `transform` con esos mismos números. Si ajustáramos el scaler con todo el dataset, estaríamos filtrando información del val al preprocesamiento (*data leakage*) y la métrica de validación quedaría inflada.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df).astype(np.float32)
X_val = scaler.transform(X_val_df).astype(np.float32)

y_train = y_train_s.to_numpy(dtype=np.float32).reshape(-1, 1)
y_val = y_val_s.to_numpy(dtype=np.float32).reshape(-1, 1)

print("X_train:", X_train.shape, "| X_val:", X_val.shape)


### 3.9 — Primero una red **lineal** (línea base)

Seguimos con `X_train`, `X_val`, `y_train` y `y_val` definidos en **3.6–3.7**.

Oculta y salida con activación **lineal**: la composición sigue siendo **afín** en las entradas (no puede “doblar” el espacio como una ReLU). Para poder usar **`binary_crossentropy`** sin forzar un sigmoide en la última capa, tratamos la salida como **logit** y usamos `from_logits=True` en la pérdida y en la métrica de accuracy.

Espera un **desempeño modesto** en validación: el fenómeno no tiene por qué ser casi lineal en estas variables.


In [ ]:
# Obtenemos el número de features
n_features = X_train.shape[1]


# 0. Declaramos semilla
tf.keras.utils.set_random_seed(101)

# 1. Creamos el modelos
modelo_lluvia_lineal = keras.Sequential(
    [
        layers.Input(shape=(n_features,), name="entrada_clima"),
        layers.Dense(16, activation="linear", name="oculta_lineal"),
        layers.Dense(1, activation="sigmoid", name="logit"),
    ],
    name="lluvia_lineal",
)

# 2. Compilamos el modelo
modelo_lluvia_lineal.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# 3. Entrenamos el modelo
hist_lluvia_lin = modelo_lluvia_lineal.fit(
    X_train,
    y_train,
    epochs=35,
    validation_data=(X_val, y_val),
    verbose=0,
)
print(
    "Lineal — última época | val_loss:", f"{hist_lluvia_lin.history['val_loss'][-1]:.4f}",
    "| val_accuracy:", f"{hist_lluvia_lin.history['val_accuracy'][-1]:.3f}",
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_lluvia_lin.history["loss"], label="train")
axes[0].plot(hist_lluvia_lin.history["val_loss"], label="val")
axes[0].set_title("Línea base — pérdida (BCE)")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Pérdida")
axes[0].legend()

axes[1].plot(hist_lluvia_lin.history["accuracy"], label="train")
axes[1].plot(hist_lluvia_lin.history["val_accuracy"], label="val")
axes[1].set_title("Línea base — accuracy")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
plt.suptitle("Red totalmente lineal sobre lluvia", y=1.02)
plt.tight_layout()
plt.show()


### 3.10 — ¿Cómo mejorar? **Funciones de activación**

Las capas lineales sólo rotan, escalan y trasladan el vector de características: **todo queda en transformaciones afines**. Si la frontera “natural” entre llueve / no llueve es **curva o en zigzag** en el espacio de entradas, hacen falta **no linealidades**.

En la práctica:

- **ReLU** (`max(0, x)`) en capas ocultas introduce “quiebres”: la red puede aproximar regiones mucho más ricas que un solo hiperplano.
- Para **clasificación binaria con etiqueta 0/1**, la salida suele pasar por una **sigmoide**: interpretamos la salida como **probabilidad** de la clase positiva y usamos `binary_crossentropy` **sin** `from_logits` (o bien mantenemos logits y `from_logits=True`).

Probemos **la misma arquitectura ancha**, pero con **ReLU** en la oculta y **sigmoide** en la salida.


In [ ]:
# 0. Declaramos semilla
tf.keras.utils.set_random_seed(123)

# 1. Creamos el modelo
modelo_lluvia_nl = keras.Sequential(
    [
        layers.Input(shape=(n_features,), name="entrada_clima"),
        layers.Dense(16, activation="relu", name="oculta_relu"),
        layers.Dense(1, activation="sigmoid", name="prob_lluvia_manana"),
    ],
    name="lluvia_relu_sigmoid",
)

# 2. Compilamos el modelo
modelo_lluvia_nl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# 3. Entrenamos el modelo
hist_lluvia_nl = modelo_lluvia_nl.fit(
    X_train,
    y_train,
    epochs=40,
    batch_size=256,
    validation_data=(X_val, y_val),
    verbose=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_lluvia_nl.history["loss"], label="train")
axes[0].plot(hist_lluvia_nl.history["val_loss"], label="val")
axes[0].set_title("ReLU + sigmoide — pérdida")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Pérdida (BCE)")
axes[0].legend()

axes[1].plot(hist_lluvia_nl.history["accuracy"], label="train")
axes[1].plot(hist_lluvia_nl.history["val_accuracy"], label="val")
axes[1].set_title("ReLU + sigmoide — accuracy")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
plt.suptitle("Misma anchura oculta, ahora con activaciones", y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(hist_lluvia_lin.history["val_accuracy"], label="val (lineal)")
ax.plot(hist_lluvia_nl.history["val_accuracy"], label="val (ReLU+sigmoide)")
ax.set_xlabel("Época")
ax.set_ylabel("Accuracy en validación")
ax.set_title("Comparación rápida en validación")
ax.legend()
plt.tight_layout()
plt.show()


## 4 — Puntos rojos y azules (dos lunas)

`make_moons` genera dos medias lunas: **no** son separables con una recta. Repetimos la idea del paso 3: **primero una red lineal** (mal desempeño), luego **ReLU + sigmoide** y curvas de entrenamiento.


In [ ]:
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=400, noise=0.18, random_state=42)
X_moon = X_moon.astype(np.float32)
y_moon = y_moon.astype(np.float32).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(6.5, 5))
m0 = (y_moon.ravel() == 0)
m1 = ~m0
ax.scatter(X_moon[m0, 0], X_moon[m0, 1], c='#DC2626', edgecolors='white', s=28, linewidths=0.4, label='Clase 0 (rojo)')
ax.scatter(X_moon[m1, 0], X_moon[m1, 1], c='#2563EB', edgecolors='white', s=28, linewidths=0.4, label='Clase 1 (azul)')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Dos lunas — no separable con una recta')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


### 4.1 — ¿Qué pueden hacer las redes con activaciones?

Resultados tipo **teorema de aproximación universal** (en formas modernas: redes poco profundas o profundas con activaciones no lineales adecuadas) dicen, en esencia, que **familias de redes con no linealidades** pueden aproximar funciones razonables si hay **suficientes neuronas** y una arquitectura adecuada — con matices de regularidad, dimensiones y costo de entrenamiento.

En el plano, las dos lunas son el ejemplo visual: **no hay una recta** que separe bien las clases. Una cadena **sólo lineal** tampoco puede arreglar eso: sigue siendo composición de mapas afines.


In [ ]:
from sklearn.model_selection import train_test_split

Xm_train, Xm_val, ym_train, ym_val = train_test_split(
    X_moon,
    y_moon,
    test_size=0.2,                  # 80% train / 20% val
    random_state=0,                 # semilla para que la partición sea reproducible
    stratify=np.ravel(y_moon),      # misma proporción de clase 0 y clase 1 en train y val
)
print("Lunas — train:", Xm_train.shape, "val:", Xm_val.shape)


### 4.2 — Línea base **lineal** en las lunas

Igual que en lluvia: oculta y salida **lineales**, BCE con **logits**.


In [ ]:
# 0. Declaramos semilla
tf.keras.utils.set_random_seed(11)

# 1. Creamos el modelo
modelo_lunas_lineal = keras.Sequential(
    [
        layers.Input(shape=(2,)),
        layers.Dense(24, activation="linear"),
        layers.Dense(1, activation="linear"),
    ],
    name="lunas_lineal",
)

# 2. Compilamos el modelo
modelo_lunas_lineal.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.02),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(threshold=0.0)],
)

# 3. Entrenamos el modelo
hist_lunas_lin = modelo_lunas_lineal.fit(
    Xm_train,
    ym_train,
    epochs=120,
    batch_size=32,
    validation_data=(Xm_val, ym_val),
    verbose=0,
)
print("Lineal lunas — val_binary_accuracy final:", f"{hist_lunas_lin.history['val_binary_accuracy'][-1]:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_lunas_lin.history["loss"], label="train")
axes[0].plot(hist_lunas_lin.history["val_loss"], label="val")
axes[0].set_title("Lunas lineales — pérdida")
axes[0].legend()
axes[1].plot(hist_lunas_lin.history["binary_accuracy"], label="train")
axes[1].plot(hist_lunas_lin.history["val_binary_accuracy"], label="val")
axes[1].set_title("Lunas lineales — accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()


### 4.4 — **Sigmoide** (binario) y activaciones ocultas

En **dos clases**, una salida escalar pasada por **sigmoide** $\sigma(z)=1/(1+e^{-z})$ manda el logit a **$(0,1)$**: se lee como $P(y=1\mid x)$. La **entropía cruzada binaria** compara esa probabilidad con la etiqueta.

En las **capas ocultas**, la **ReLU** introduce piezas afines por regiones: al componer varias capas, aparecen **fronteras** mucho más flexibles. Ahora sí entrenamos la red “de manual” para las lunas.


In [ ]:
# 0. Declaramos semilla
tf.keras.utils.set_random_seed(7)

# 1. Creamos el modelo
modelo_lunas = keras.Sequential(
    [
        layers.Input(shape=(2,)),
        layers.Dense(24, activation="relu"),
        layers.Dense(24, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="dos_lunas",
)

# 2. Compilamos el modelo
modelo_lunas.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# 3. Entrenamos el modelo
hist_lunas = modelo_lunas.fit(
    Xm_train,
    ym_train,
    epochs=200,
    batch_size=32,
    validation_data=(Xm_val, ym_val),
    verbose=0,
)
print("ReLU lunas — val_accuracy final:", f"{hist_lunas.history['val_accuracy'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_lunas.history["loss"], label="train")
axes[0].plot(hist_lunas.history["val_loss"], label="val")
axes[0].set_title("Dos lunas — pérdida")
axes[0].legend()
axes[1].plot(hist_lunas.history["accuracy"], label="train")
axes[1].plot(hist_lunas.history["val_accuracy"], label="val")
axes[1].set_title("Dos lunas — accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(hist_lunas_lin.history["val_binary_accuracy"], label="val (lineal)")
ax.plot(hist_lunas.history["val_accuracy"], label="val (ReLU+sigmoide)")
ax.set_xlabel("Época")
ax.set_ylabel("Accuracy val")
ax.set_title("Lunas: comparación en validación")
ax.legend()
plt.tight_layout()
plt.show()


### 4.5 — ¿Y si la curva de pérdida se ve así? (overfitting)

En las gráficas de arriba notarás que el `train_loss` **sigue bajando**, pero el `val_loss` **empieza a subir** alrededor de la época 100 (con picos al final). La accuracy de val se atora ~94 % mientras la de train se va a 99 %. Eso es **sobreajuste** clásico: el modelo se está aprendiendo el ruido del train en vez de generalizar.

Cosas que puedes probar, de más simple a más sofisticado:

- **Menos épocas.** 200 es exceso para 320 puntos. Bajar a ~80 ya alcanza el plateau sin la subida del val.
- **Early stopping.** Que Keras pare solo cuando `val_loss` deja de mejorar:
  ```python
  cb = keras.callbacks.EarlyStopping(
      monitor='val_loss', patience=15, restore_best_weights=True
  )
  modelo_lunas.fit(..., callbacks=[cb])
  ```
- **Reducir capacidad.** Dos capas de 24 ReLU es bastante para dos lunas. Prueba `Dense(16)` o una sola capa oculta.
- **Regularización.** Añade `Dropout(0.2)` entre capas, o `kernel_regularizer=keras.regularizers.l2(1e-3)` en las `Dense`.
- **Bajar el `learning_rate`** (de 0.01 a 0.003) para que no oscile al final del entrenamiento.
- **Más datos / menos ruido.** Subir `n_samples` en `make_moons` o bajar `noise` también ayuda — aquí lo controlas tú porque es sintético.

Lo más vistoso es el **early stopping**: una sola línea y la curva de val deja de subir.


In [ ]:
# Frontera aproximada: rejilla y contorno donde la sigmoide cruza 0.5
pad = 0.35
x_min, x_max = float(X_moon[:, 0].min() - pad), float(X_moon[:, 0].max() + pad)
y_min, y_max = float(X_moon[:, 1].min() - pad), float(X_moon[:, 1].max() + pad)
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 160), np.linspace(y_min, y_max, 160))
grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
Z = modelo_lunas.predict(grid, verbose=0).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.contourf(xx, yy, Z, levels=25, cmap='RdBu_r', alpha=0.55)
ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=1.2)
m0 = (y_moon.ravel() == 0)
m1 = ~m0
ax.scatter(X_moon[m0, 0], X_moon[m0, 1], c='#DC2626', edgecolors='white', s=22, linewidths=0.3, label='Clase 0')
ax.scatter(X_moon[m1, 0], X_moon[m1, 1], c='#2563EB', edgecolors='white', s=22, linewidths=0.3, label='Clase 1')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Región de decisión (probabilidad de clase 1)')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## 5 — Tres categorías y **softmax**

Con **más de dos clases**, una sola sigmoide ya no basta: necesitamos un vector de **probabilidades** que sume 1. La **softmax** transforma logits $(z_1,\ldots,z_K)$ en

$$
p_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}.
$$

La pérdida típica es **entropía cruzada categórica** (aquí `sparse_categorical_crossentropy`: las etiquetas van como **entero** de clase 0, 1 o 2). La última capa es `Dense(3, activation="softmax")`.


### Aparte: ¿cómo entra la probabilidad en una red? (entropía cruzada)

Hasta aquí hemos hablado del modelo como si fuera una **calculadora**: entran números, sale un número (estatura, lluvia/no lluvia, clase). Pero en clasificación pasa algo más bonito: **la red no escupe una clase**, escupe una **distribución de probabilidad** sobre las clases. La entropía cruzada (`binary_crossentropy`, `sparse_categorical_crossentropy`) es la pérdida que **compara dos distribuciones de probabilidad**: la que predice la red y la que dicen los datos.

#### 1) La salida de la red ES una distribución

- **Sigmoide** (binario): la salida es un número entre 0 y 1, que se lee como $P(y=1 \mid x)$. La probabilidad de la otra clase es $1 - P(y=1 \mid x)$. Las dos suman 1 → es una distribución de probabilidad sobre **dos clases**.
- **Softmax** (multiclase): la capa de salida tiene $K$ neuronas (una por clase). Softmax garantiza que las $K$ salidas sean **positivas y sumen 1**:
  $$
  p_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}
  $$
  Por ejemplo, para 3 clases podría salir $[0.7, 0.2, 0.1]$ → "70 % seguro de clase 0, 20 % de clase 1, 10 % de clase 2".

Por eso decimos que la red **no clasifica de un sí/no rotundo**, sino que **estima qué tan creíble es cada opción**. La predicción "dura" se obtiene tomando el máximo, pero internamente la red razona en probabilidades.

#### 2) La etiqueta real también es una distribución (extrema)

Si el dato tiene etiqueta clase 1, podemos representarla como una distribución que pone **toda su masa** en esa clase:

| | Clase 0 | Clase 1 | Clase 2 |
|---|---|---|---|
| Etiqueta real (one-hot) | 0 | **1** | 0 |
| Predicción de la red | 0.7 | 0.2 | 0.1 |

Es una distribución *certeza absoluta*: el dato es de clase 1 con probabilidad 1 y de las demás con probabilidad 0. La pregunta natural es: **¿qué tan lejos está la predicción de esta certeza?**

#### 3) La entropía cruzada mide esa "distancia"

Para una sola observación con clase verdadera $c$, la entropía cruzada se reduce a:

$$
\mathrm{CE} = -\log\big(p_c\big)
$$

donde $p_c$ es **la probabilidad que la red le dio a la clase correcta**. Léela así: "la pérdida es el log negativo de qué tanto le atinaste a la verdadera".

- Si la red dijo $p_c = 1.0$ (perfecta confianza en la correcta) → $-\log(1) = 0$. **Pérdida cero**.
- Si dijo $p_c = 0.7$ → $-\log(0.7) \approx 0.36$. **Pérdida pequeña**, le atinó razonablemente.
- Si dijo $p_c = 0.2$ → $-\log(0.2) \approx 1.61$. **Pérdida alta**, casi descartó la correcta.
- Si dijo $p_c = 0.001$ → $-\log(0.001) \approx 6.9$. **Pérdida brutal**, prácticamente le pegó al lado.
- Si dijo $p_c \to 0$ → $-\log(0) \to \infty$. **Pérdida infinita** — la red estaba "absolutamente segura" de algo falso.

Esa es la magia: la entropía cruzada **castiga exponencialmente la sobre-confianza equivocada**. No es lo mismo equivocarse "dudando" (decir 40 %) que equivocarse "convencido" (decir 1 %).

#### 4) Para el dataset entero, se promedia

La pérdida que reporta Keras es el promedio de $-\log(p_c)$ sobre todas las filas del batch (o de la época):

$$
\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \log\big(p_{c_i}^{(i)}\big)
$$

donde $p_{c_i}^{(i)}$ es la probabilidad que la red le asignó a la clase verdadera de la fila $i$. Es **un solo número** que resume "qué tan sorprendida quedó la red, en promedio, por las etiquetas reales".

#### 5) ¿Por qué NO usar MSE en clasificación?

Si tratáramos las probabilidades como números y midiéramos $(y - \hat{y})^2$:

- El MSE penaliza poco a una predicción de 0.01 cuando la respuesta correcta era 1 (error cuadrado de ~0.98). La entropía cruzada lo castiga con $-\log(0.01) = 4.6$, **mucho más fuerte**.
- Los gradientes del MSE con sigmoide se desvanecen cuando la predicción se acerca a 0 o 1 (saturación). Los de cross-entropy se mantienen útiles incluso cerca de los extremos.
- En clasificación queremos que la red **se confíe correctamente**; cross-entropy empuja exactamente eso.

#### 6) Las dos variantes que verás en este notebook

- **`binary_crossentropy`**: para clasificación binaria (lluvia, lunas). Espera salida con `sigmoid` y etiquetas en `{0, 1}`.
- **`sparse_categorical_crossentropy`**: para multiclase. Espera salida con `softmax` y etiquetas como **enteros** de clase (`0, 1, 2`). La versión "no sparse" (`categorical_crossentropy`) espera las etiquetas en formato **one-hot** (`[0, 0, 1]`). Hacen lo mismo, solo cambia el formato de la etiqueta.

#### Lectura intuitiva para llevarte

- Cada predicción de la red = **una apuesta probabilística** sobre las clases.
- La entropía cruzada = **qué tan sorprendida queda la red** cuando se revela la respuesta correcta.
- Entrenar = **enseñarle a no sorprenderse** (a poner alta probabilidad en la clase verdadera y baja en las demás).

Por eso decimos que la red "razona en probabilidades" — no porque alguien se lo programe, sino porque la pérdida (cross-entropy) está construida para que **solo bajándola tiene sentido producir distribuciones bien calibradas**.


### 5.1 — Datos sintéticos con **tres** grupos

`make_blobs` en el plano: tres manchas de puntos con etiquetas 0, 1 y 2.


In [ ]:
from sklearn.datasets import make_blobs

Xb, yb = make_blobs(
    n_samples=600,
    centers=3,
    cluster_std=0.85,
    random_state=2,
)
Xb = Xb.astype(np.float32)
yb = yb.astype(np.int32)

Xb_train, Xb_val, yb_train, yb_val = train_test_split(
    Xb, yb,
    test_size=0.25,    # 75% train / 25% val
    random_state=3,    # semilla para reproducir la misma partición
    stratify=yb,       # cada clase (0, 1, 2) queda igual de representada en train y val
)

fig, ax = plt.subplots(figsize=(6, 5))
for k, color in zip([0, 1, 2], ["#DC2626", "#2563EB", "#16A34A"]):
    m = yb == k
    ax.scatter(Xb[m, 0], Xb[m, 1], c=color, s=22, alpha=0.65, label=f"Clase {k}")
ax.set_title("Tres blobs en el plano")
ax.legend()
plt.tight_layout()
plt.show()
print("Formas train:", Xb_train.shape, yb_train.shape)


### 5.2 — Modelo con **softmax** en la salida

Tres neuronas de salida + softmax ⇒ tres probabilidades que suman 1.


In [ ]:
# 0. Declaramos semilla
tf.keras.utils.set_random_seed(5)

# 1. Creamos el modelo
modelo_tres = keras.Sequential(
    [
        layers.Input(shape=(2,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax", name="probs_tres_clases"),
    ],
    name="tres_clases_softmax",
)

# 2. Compilamos el modelo
modelo_tres.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# 3. Entrenamos el modelo
hist_tres = modelo_tres.fit(
    Xb_train,
    yb_train,
    epochs=80,
    batch_size=32,
    validation_data=(Xb_val, yb_val),
    verbose=0,
)
print("Val accuracy final:", f"{hist_tres.history['val_accuracy'][-1]:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_tres.history["loss"], label="train")
axes[0].plot(hist_tres.history["val_loss"], label="val")
axes[0].set_title("Tres clases — pérdida")
axes[0].legend()
axes[1].plot(hist_tres.history["accuracy"], label="train")
axes[1].plot(hist_tres.history["val_accuracy"], label="val")
axes[1].set_title("Tres clases — accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()


## 6 — Hiperparámetros que sí puedes tocar en este cuaderno

| Idea | Dónde aparece | Qué pasa si lo mueves |
|------|----------------|------------------------|
| **`N_NEURONAS`** (o listas `Dense(k)`) | Arquitectura | Más neuronas → más capacidad para curvas; también más riesgo de sobreajuste si el set es chico. |
| **`epochs`** | `model.fit(...)` | Más vueltas al dataset: a veces baja más la pérdida; a veces el **val** empeora (sobreentrenamiento). |
| **`batch_size`** | `model.fit(...)` | Lotes chicos: gradientes más ruidosos, pasos más frecuentes; lotes grandes: más estable, menos actualizaciones por época. |
| **`learning_rate`** | `Adam(learning_rate=...)` | Grande: aprende rápido pero puede oscilar o diverger; chico: más suave, a veces muy lento. |
| **`validation_split`** | `model.fit(...)` | Fracción reservada para **val** en cada época (no entrena sobre ella, sólo la mide). |
| **Número de capas ocultas** | Más bloques `Dense` + activación | Más no linealidades compuestas; también más parámetros que ajustar. |

El **optimizador** (`Adam`, `SGD`, …) y la **función de pérdida** también son elecciones de diseño, pero en clasificación binaria con sigmoide suele quedarse en `binary_crossentropy`.


**Cierre.** Recorrimos el mismo guion dos veces: **línea base lineal** (limitada), luego **activaciones** (ReLU en ocultas, **sigmoide** en binario). Vimos que las redes con no linealidades pueden **aproximar fronteras** difíciles en el plano. Cerramos con **tres clases**, **softmax** y entropía cruzada categórica **sparse**: el puente natural hacia más etiquetas y hacia embeddings en NLP.
